# FLUX Spectral Analysis — Exploration

Interactive notebook for exploring cached radial power spectra computed by the
analysis pipeline.  Run `python scripts/run_analysis.py` first to populate the
`data/features/` directory with `.npz` cache files.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import yaml
import sys
from pathlib import Path

%matplotlib inline
plt.rcParams['figure.dpi'] = 100
print('Imports OK')

In [ ]:
# Add project root to path so src/ modules are importable
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
print(f'Repo root: {REPO_ROOT}')

# Load experiment config
config_path = REPO_ROOT / 'configs' / 'experiment.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

features_dir = REPO_ROOT / cfg['output']['features_dir']
print(f'Features dir: {features_dir}')
print(f'Config: {cfg["data"]}')

## Load spectra

Load the cached `.npz` files produced by `run_analysis.py`.  Each file contains
a `spectra` array of shape `(N, R)` where N = number of images and R = number
of radial frequency bins.

In [ ]:
real_cache = features_dir / 'real_spectra.npz'
if real_cache.exists():
    real_spectra = np.load(real_cache)['spectra']
    print(f'Real spectra shape: {real_spectra.shape}')
else:
    print(f'Cache not found: {real_cache}')
    print('Run: python scripts/run_analysis.py')
    real_spectra = None

gen_spectra_dict = {}
for group_name in cfg['data']['generated_dirs']:
    cache_path = features_dir / f'{group_name}_spectra.npz'
    if cache_path.exists():
        gen_spectra_dict[group_name] = np.load(cache_path)['spectra']
        print(f'{group_name} spectra shape: {gen_spectra_dict[group_name].shape}')
    else:
        print(f'Cache not found for {group_name}: {cache_path}')

## Quick visualizations

Plot the mean radial power spectra on a log-log scale to inspect the spectral
slope and compare real vs. generated images.

In [ ]:
EPSILON = 1e-12
COLORS = {
    'real': '#2c7bb6',
    'klein_4b_distilled': '#d7191c',
    'klein_4b_base': '#fdae61',
}

fig, ax = plt.subplots(figsize=(10, 5))

if real_spectra is not None:
    real_mean = real_spectra.mean(axis=0)
    real_std  = real_spectra.std(axis=0)
    freqs = np.arange(len(real_mean))
    log_mean = np.log10(real_mean + EPSILON)
    log_upper = np.log10(real_mean + real_std + EPSILON)
    log_lower = np.log10(np.maximum(real_mean - real_std, EPSILON))
    color = COLORS.get('real', '#888888')
    ax.plot(freqs, log_mean, label='real', color=color, linewidth=1.5)
    ax.fill_between(freqs, log_lower, log_upper, alpha=0.15, color=color)

for group_name, gen_spectra in gen_spectra_dict.items():
    gen_mean = gen_spectra.mean(axis=0)
    gen_std  = gen_spectra.std(axis=0)
    freqs = np.arange(len(gen_mean))
    log_mean = np.log10(gen_mean + EPSILON)
    log_upper = np.log10(gen_mean + gen_std + EPSILON)
    log_lower = np.log10(np.maximum(gen_mean - gen_std, EPSILON))
    color = COLORS.get(group_name, '#d7191c')
    ax.plot(freqs, log_mean, label=group_name, color=color, linewidth=1.5)
    ax.fill_between(freqs, log_lower, log_upper, alpha=0.15, color=color)

ax.set_xlabel('Frequency bin (radial)')
ax.set_ylabel('log₁₀(Power)')
ax.set_title('Mean Radial Power Spectra — Real vs. Generated')
ax.legend()
ax.spines['right'].set_visible(False)
ax.spines['top'].set_visible(False)
plt.tight_layout()
plt.show()